# Everything about dictionaries, sets, lookups, and hashing

By the end you will know:

- why `dict` and `set` lookups are fast
- when to use a dict, a set, or a list
- how to count every word in 500 book titles
- the exam patterns that reuse these ideas

You only need: variables, `if`, `for`, strings, and lists.


## 1. The problem a list cannot solve

Imagine a phone book as a **list of pairs**. To find Chidi you start at Ada, then Bola, then Chidi. If Chidi is last among 10,000 names, you check 10,000 times. That is a **linear search**, written **O(n)** — work grows with the size of the list.


In [37]:
phones_list = [
    ("Ada", "0801"),
    ("Bola", "0802"),
    ("Chidi", "0803"),
]

# Hunt for Chidi: walk from the start
number = None
for name, num in phones_list:
    if name == "Chidi":
        number = num
        break
print("List hunt found:", number)


List hunt found: 0803


The same data as a **dictionary**. Python does not walk the names. It turns `"Chidi"` into a locker number and opens that locker. 3 names or 3 million names: looking up Chidi costs about the same. That is **O(1)** — constant time, about one step.


In [ ]:
phones = {
    "Ada": "0801",
    "Bola": "0802",
    "Chidi": "0803",
}
print(phones["Chidi"])       # one jump, not a hunt
print("Chidi" in phones)     # True — also a hash jump


0803
True


## 2. Dictionaries from zero

A **dictionary** (`dict`) stores **pairs**: a **key** (the label) and a **value** (the thing in the locker).


In [ ]:
# empty dict
d = {}
d = dict()

# with data
book = {
    "title": "Things Fall Apart",
    "author": "Chinua Achebe",
    "year": 1958,
}
book


{'title': 'Things Fall Apart', 'author': 'Chinua Achebe', 'year': 1958}

### Read, write, add, overwrite

`book["price"]` raises **KeyError** if the key is missing. Safe read: `book.get("price")` returns `None`, or `book.get("price", 0)` returns `0`.


In [39]:
print(book["title"])           # read
book["pages"] = 209            # add a new key
book["year"] = 1959  
book["price"] = 200   
print("author" in book)        # True — "is this KEY here?" FAST
print("price" in book)         # False
print(book.get("price"))       # None — no crash
print(book.get("price", 0))    # 0 — default
book


Things Fall Apart
True
True
200
200


{'title': 'Things Fall Apart',
 'author': 'Chinua Achebe',
 'year': 1959,
 'pages': 209,
 'price': 200}

### The counting pattern (you will use this forever)

`get(word, 0)` means: “give me the current count, or 0 if I have never seen this word.” Then add 1.


In [5]:
counts = {}
for word in ["the", "cat", "the", "hat"]:
    if word in counts:
        counts[word] += 1
    else:
        counts[word] = 1
print("long form:", counts)

counts = {}
for word in ["the", "cat", "the", "hat"]:
    counts[word] = counts.get(word, 0) + 1
print("short form:", counts)


long form: {'the': 2, 'cat': 1, 'hat': 1}
short form: {'the': 2, 'cat': 1, 'hat': 1}


### Delete


In [41]:
demo = {"a": 1, "b": 2, "c": 3}
del demo["a"]                  # gone, or KeyError if missing
b_val = demo.pop("b")          # remove AND return the value
demo.pop("missing", None)      # safe pop: no crash
print("popped b =", b_val)
print("left:", demo)
demo.clear()
print("after clear:", demo)


popped b = 2
left: {'c': 3}
after clear: {}


## 3. Sets from zero

A **set** is a bag of unique items. No duplicates. Do not treat it like a numbered list.

Think of a set as a dictionary that only has keys, no values. Same hashing. Same speed. Question it answers: *“Is this thing in the collection?”*

> **Trap:** `{}` is an empty **dictionary**. An empty set is always `set()`.


In [7]:
empty = set()                  # NOT {}
print("type of {} :", type({}))
print("type of set():", type(set()))

vowels = {"a", "e", "i", "o", "u"}
vowels.add("y")
vowels.discard("z")            # safe: does nothing if missing
print("e" in vowels)           # True — FAST, same hashing as dict
print(vowels)


type of {} : <class 'dict'>
type of set(): <class 'set'>
True
{'a', 'e', 'y', 'o', 'i', 'u'}


### Why sets exist — uniqueness


In [8]:
words = ["the", "cat", "the", "hat", "the"]
unique = set(words)
print(unique)
print("list length:", len(words), "| set length:", len(unique))


{'the', 'cat', 'hat'}
list length: 5 | set length: 3


### Set math in plain English

- **union** `|` = everyone (in A or B or both)
- **intersection** `&` = the overlap
- **difference** `-` = in A but not B
- **symmetric difference** `^` = in one, not both


In [9]:
a = {1, 2, 3}
b = {3, 4, 5}
print("union        a | b :", a | b)
print("intersection a & b :", a & b)
print("difference   a - b :", a - b)
print("symmetric    a ^ b :", a ^ b)


union        a | b : {1, 2, 3, 4, 5}
intersection a & b : {3}
difference   a - b : {1, 2}
symmetric    a ^ b : {1, 2, 4, 5}


## 4. Hashing: why lookups are fast

This is the heart of the topic.

A list is a queue. To find `"cat"` you ask person 0, person 1, person 2…

A dict/set is a wall of lockers. Python runs a **hash function** on your key, turns that integer into a locker index (think: `hash(key) % number_of_lockers`), and **jumps** there.


In [10]:
print(hash("cat"))
print(hash("cat"))     # SAME number again, in this Python process
print(hash("hat"))


-2155170339348579930
-2155170339348579930
8542694521201701583


### What “O(1) lookup” really means

| Action | list / tuple | dict / set |
| --- | --- | --- |
| Is `"cat"` here? | Scan all items. **O(n)** | Hash and jump. **O(1) average** |
| Get item at position 0 | **O(1)** — lists are great at this | Dicts are not numbered this way |
| Add one item | O(1) at the end | O(1) average |
| Remove a known key | O(n) from the middle (items slide) | O(1) average |

**Average O(1)** is the honest phrase. Python resizes the locker wall when it gets crowded, so this stays true as the dict grows.

**Collisions:** two different keys can hash to the same locker. Python keeps a small chain or finds the next slot. You still get the right value. If *every* key collided (almost never), lookup would degrade toward O(n).

**Exam sentence:** *Dictionary and set lookups are average O(1) because keys are hashed to a bucket index. Lists are O(n) because membership walks the whole collection. Use a dict when each key maps to a value; use a set when you only need uniqueness or membership.*


### Tiny proof you can run

The needle is the **last** item — worst case for the list, still one jump for the set.


In [11]:
import time

n = 80_000
lst = list(range(n))
st = set(range(n))
needle = n - 1

t0 = time.perf_counter()
_ = needle in lst
t1 = time.perf_counter()
_ = needle in st
t2 = time.perf_counter()

print(f"list membership: {t1 - t0:.6f} seconds")
print(f"set  membership: {t2 - t1:.6f} seconds")
print("The set stays cheap as n grows. The list does not.")


list membership: 0.002539 seconds
set  membership: 0.000132 seconds
The set stays cheap as n grows. The list does not.


The hash must be **stable** for the object. If a key could change after you insert it, its locker number would be wrong later. So Python only allows **hashable** (usually **immutable**) keys. See section 8.


## 5. When to use dict, set, or list

**Use a dict when**

- you look up data attached to a name: word → count, id → student
- you need “for this key, give me its value”
- you are grouping: category → list of items
- you are caching: input → already-computed answer

**Use a set when**

- you only care “is it here?”
- you need unique items
- you need overlap / difference of two collections
- you are marking “already visited”

**Use a list when**

- order matters (first, second, third)
- duplicates are allowed and meaningful
- you need item #0, #1, #2
- you will loop through everything anyway

If you catch yourself writing `if x in my_list` **inside a loop** over lots of data, you probably wanted a set or a dict.


In [12]:
user_ids = [10, 20, 30, 20, 40, 10]

# SLOW idea: membership on a list (scans every time)
seen_list = []
dupes_via_list = []
for user_id in user_ids:
    if user_id in seen_list:      # O(n) scan
        dupes_via_list.append(user_id)
    else:
        seen_list.append(user_id)

# FAST idea: same question on a set
seen = set()
dupes_via_set = []
for user_id in user_ids:
    if user_id in seen:           # O(1) hash jump
        dupes_via_set.append(user_id)
    else:
        seen.add(user_id)

print("duplicates:", dupes_via_set)


duplicates: [20, 10]


## 6. Every dictionary move you need

Memorise these. They cover almost every exam question.

From Python 3.7 onward, dicts remember **insertion order**. That is not the same as **sorted**.


In [ ]:
d = {"a": 1, "b": 2}

print(d["a"])                 # 1, or KeyError
print(d.get("z"))             # None
print(d.get("z", 0))          # 0
print(d.setdefault("w", 3))    # if 'w' missing, put 3, then return it

print("keys  :", list(d.keys()))
print("values:", list(d.values()))
print("items :", list(d.items()))

print("a" in d)               # True  — tests KEYS, not values
print(1 in d)                 # False
print(1 in d.values())        # True — this IS a scan, slower

d.update({"b": 9, "d": 4})    # merge in
e = {**d, "x": 1}             # new dict, copy + extra
f = d | {"z": 0}              # Python 3.9+ merge
print("merged f:", f)
print("len:", len(d))


1
None
0
keys  : ['a', 'b', 'w']
values: [1, 2, 3]
items : [('a', 1), ('b', 2), ('w', 3)]
True
False
True
merged f: {'a': 1, 'b': 9, 'w': 3, 'd': 4, 'z': 0}
len: 4


### Loop the right way


In [14]:
d = {"title": "Americanah", "year": 2013}

print("--- keys ---")
for key in d:
    print(key, d[key])

print("--- pairs (preferred) ---")
for key, value in d.items():
    print(key, value)

print("--- values ---")
for value in d.values():
    print(value)


--- keys ---
title Americanah
year 2013
--- pairs (preferred) ---
title Americanah
year 2013
--- values ---
Americanah
2013


### Sort a dict by value (top words, leaderboards)

`counts.items()` gives pairs. `key=lambda kv: kv[1]` means “sort using the second thing in the pair” (the count). `reverse=True` means biggest first.


In [15]:
counts = {"the": 225, "of": 160, "a": 49}
ranked = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
ranked


[('the', 225), ('of', 160), ('a', 49)]

## 7. Every set move you need


In [16]:
s = {1, 2, 3}
s.add(4)
s.update([4, 5, 6])    # add many
s.discard(9)           # no error
s.remove(1)            # error if missing
print("set now:", s)
print("4 in s:", 4 in s)
print("len:", len(s))

a = {1, 2, 3}
b = {3, 4}
print("union names:", a.union(b))
print("subset?", {1, 2}.issubset(a))
print("disjoint?", a.isdisjoint({9, 10}))


set now: {2, 3, 4, 5, 6}
4 in s: True
len: 5
union names: {1, 2, 3, 4}
subset? True
disjoint? True


Building from a list is the usual way to unique-ify. A raw set may change order. `dict.fromkeys` keeps **first-seen** order.


In [17]:
names = ["Ada", "Ada", "Bola", "Chidi", "Bola"]
print("set (unique, order may change):", list(set(names)))
print("keep first-seen order:        ", list(dict.fromkeys(names)))


set (unique, order may change): ['Ada', 'Chidi', 'Bola']
keep first-seen order:         ['Ada', 'Bola', 'Chidi']


## 8. The hashing rules (what can be a key)

A dict key or set item must be **hashable**. Beginner version: Python must be able to run `hash(x)` without crashing, and that number must never change while `x` lives in the table.

| Type | Hashable? | OK as dict key / set item? |
| --- | --- | --- |
| `int`, `float`, `bool` | yes | yes |
| `str` | yes | yes — most common key |
| `tuple` of hashables, e.g. `(1, "a")` | yes | yes — great for pair-keys |
| `frozenset` | yes | yes |
| `None` | yes | yes |
| `list` | no | TypeError |
| `dict` | no | TypeError |
| `set` | no | TypeError |
| tuple that contains a list, e.g. `(1, [2])` | no | TypeError |

Why lists are banned: lists can change. If Python put a list in locker 5, then you `.append`, the hash would want a different locker. Tuples cannot be changed, so they are safe.

**Pair-as-key trick:** count pairs or coordinates with `counts[(r, c)] = counts.get((r, c), 0) + 1`.

`True` hashes like `1`, and `False` like `0`. So `{1: "one", True: "true"}` has *one* key. Weird, but good for quizzes.


In [18]:
d = {}
d["ok"] = 1
d[3] = 2
d[(1, 2)] = 3                 # tuple key: fine
print(d)

# Prove lists are banned — catch the error so Run All still works
try:
    d[[1, 2]] = 4
except TypeError as err:
    print("list as key:", err)

try:
    bad = {[1, 2]}
except TypeError as err:
    print("list in set:", err)

print("hashable str? ", end="")
print(hash("hello") is not None)

def can_be_key(value):
    try:
        hash(value)
        return True
    except TypeError:
        return False

print("can_be_key('hello') :", can_be_key("hello"))
print("can_be_key([1, 2])  :", can_be_key([1, 2]))
print("can_be_key((1, 2))  :", can_be_key((1, 2)))
print("True vs 1 as keys   :", {1: "one", True: "true"})


{'ok': 1, 3: 2, (1, 2): 3}
list as key: unhashable type: 'list'
list in set: unhashable type: 'list'
hashable str? True
can_be_key('hello') : True
can_be_key([1, 2])  : False
can_be_key((1, 2))  : True
True vs 1 as keys   : {1: 'true'}


## 9. Loops, unpacking, comprehensions


In [19]:
names = ["ada", "bola"]
lengths = {n: len(n) for n in names}
print("dict comprehension:", lengths)

flipped = {v: k for k, v in lengths.items()}
print("flipped (values must be unique AND hashable):", flipped)

words = ["The", "the", "Cat"]
lower = {w.lower() for w in words}
print("set comprehension:", lower)


dict comprehension: {'ada': 3, 'bola': 4}
flipped (values must be unique AND hashable): {3: 'ada', 4: 'bola'}
set comprehension: {'the', 'cat'}


### Nested dict (a dict inside a dict)

Two jumps: first key, then inner key. Still hashing, twice.


In [20]:
library = {
    "Things Fall Apart": {"author": "Achebe", "year": 1958},
    "Americanah": {"author": "Adichie", "year": 2013},
}
print(library["Americanah"]["author"])


Adichie


### Dict of lists (grouping)


In [21]:
pairs = [
    ("Americanah", "Adichie"),
    ("Half of a Yellow Sun", "Adichie"),
    ("Things Fall Apart", "Achebe"),
]
by_author = {}
for title, author in pairs:
    if author not in by_author:
        by_author[author] = []
    by_author[author].append(title)
by_author


{'Adichie': ['Americanah', 'Half of a Yellow Sun'],
 'Achebe': ['Things Fall Apart']}

## 10. Helpers: Counter, defaultdict, frozenset

You can live without these. They just shorten the counting and grouping patterns.

**frozenset** is a set that cannot change, so it *can* be a dict key.


In [22]:
from collections import Counter, defaultdict

c = Counter("mississippi")
print("count of s:", c["s"])
print("most common:", c.most_common(2))

counts = defaultdict(int)      # missing key -> 0
for w in ["the", "the", "cat"]:
    counts[w] += 1             # no .get needed
print(dict(counts))

groups = defaultdict(list)     # missing key -> new list
groups["Adichie"].append("Americanah")
print(dict(groups))

cache = {}
cache[frozenset(["eat", "tea", "ate"])] = "anagram family"
print(cache)


count of s: 4
most common: [('i', 4), ('s', 4)]
{'the': 2, 'cat': 1}
{'Adichie': ['Americanah']}
{frozenset({'eat', 'tea', 'ate'}): 'anagram family'}


## 11. Traps that catch beginners

1. `{}` is a dict, not a set.
2. `in` on a dict checks **keys**, not values.
3. Do not loop and `del` keys from the same dict. Build a list of keys to delete, or build a new dict.
4. `d.copy()` is shallow: inner lists/dicts are still shared.
5. A list is not a valid key. Convert to tuple: `tuple(my_list)`.
6. `set1 + set2` is illegal. Use `set1 | set2`.
7. Hash values change between Python processes. Never save `hash(x)` to a file expecting it to match tomorrow. Save the key itself.
8. `for k, v in d` is wrong. Use `d.items()`.
9. Checking `x in some_list` inside a tight loop is the classic accidental **O(n²)** bug. Convert to a set first.
10. Equality of dicts ignores order: `{"a":1,"b":2} == {"b":2,"a":1}` is True.


In [23]:
print({"a": 1, "b": 2} == {"b": 2, "a": 1})

d = {"a": 1, "b": 2}
try:
    for k, v in d:
        print(k, v)
except ValueError as err:
    print("for k, v in d ->", err)
    print("fix: for k, v in d.items()")

try:
    print({1, 2} + {3, 4})
except TypeError as err:
    print("set + set ->", err)
    print("fix:", {1, 2} | {3, 4})


True
for k, v in d -> not enough values to unpack (expected 2, got 1)
fix: for k, v in d.items()
set + set -> unsupported operand type(s) for +: 'set' and 'set'
fix: {1, 2, 3, 4}


## 12. Project: count title words across 500 books

This is the practical test of the whole topic.

**Data structures**

```text
counts = {"the": 225, "of": 160, ...}   # dict: word -> how many times
unique = {"the", "of", "a", ...}        # set: words only, no counts
```

After 500 titles you do **not** store 225 copies of the word `"the"`. You store **one key** `"the"` and the number **225**. That is the point of a dictionary.

**Algorithm**

1. Start with empty `counts = {}` and empty `unique = set()`.
2. For each title in the 500…
3. Split the title into words.
4. Clean each word: lowercase, strip punctuation.
5. `unique.add(word)` — set ignores it if already there.
6. `counts[word] = counts.get(word, 0) + 1` — dict lookup by hash, add one.

**Why not a list of pairs** that you search every time? Each new word would scan the growing list. Cost blows up as the catalogue grows. The dict stays cheap: time is about **total words processed**.

**Why a set as well?**

- Dict answers “how many?”
- Set answers “how many *different* words?” and “is this a word we know?”

In this project `len(counts)` and `len(unique)` match, because every counted word is also in the set. They are still different tools.

**Teacher-ready sentence**

> I used a hash map (Python dict) from word → integer count. Each title is tokenised, each token is hashed, and the bucket for that word is incremented. Average update and lookup are O(1), so 500 books cost time proportional to the total number of words, not to the number of distinct words squared. I also kept a set of distinct tokens for membership and uniqueness. A list would make each membership test O(n) and would be the wrong structure.


### Build a catalogue of 500 unique titles

(The same generator as `books_catalogue.py`. Run this cell first.)


In [24]:
FAMOUS = [
    "Pride and Prejudice", "To Kill a Mockingbird", "The Great Gatsby",
    "One Hundred Years of Solitude", "The Catcher in the Rye", "Beloved",
    "Things Fall Apart", "The God of Small Things", "Half of a Yellow Sun",
    "Americanah", "Purple Hibiscus", "The Famished Road", "Stay with Me",
    "My Sister the Serial Killer", "The Joys of Motherhood", "Nervous Conditions",
    "So Long a Letter", "Wizard of the Crow", "Petals of Blood",
    "A Grain of Wheat", "Weep Not Child", "The River Between",
    "Death and the King's Horseman", "The Lion and the Jewel", "Arrow of God",
    "No Longer at Ease", "Anthills of the Savannah", "The Palm Wine Drinkard",
    "The Fishermen", "The Death of Vivek Oji", "Freshwater",
    "Under the Udala Trees", "The Girl with the Louding Voice",
    "An Orchestra of Minorities", "Homegoing", "The Underground Railroad",
    "Song of Solomon", "The Bluest Eye", "The Color Purple",
    "Their Eyes Were Watching God", "Between the World and Me",
    "Invisible Man", "Native Son", "I Know Why the Caged Bird Sings",
    "Kindred", "Parable of the Sower", "The Fifth Season",
    "The City We Became", "Who Fears Death", "Rosewater",
    "Akata Witch", "White Teeth", "On Beauty", "Normal People",
    "The Vanishing Half", "Wahala", "A Spell of Good Things",
    "Midnight's Children", "The English Patient", "A Passage to India",
    "Long Walk to Freedom", "Born a Crime", "Cry the Beloved Country",
    "Disgrace", "There There", "The Night Watchman",
]

STARTERS = [
    "The Last", "A Short", "Hidden", "Broken", "Silent", "Golden", "Wild",
    "Forgotten", "Secret", "Bright", "Dark", "Small", "Great", "Quiet",
    "Loud", "Old", "New", "Lost", "Found", "Red", "Blue", "Green",
    "River", "Desert", "City", "Village", "Night", "Morning", "Winter",
    "Summer", "Autumn", "Spring",
]
MIDDLES = [
    "Song", "Story", "House", "Road", "River", "Garden", "Market", "Child",
    "Mother", "Father", "King", "Queen", "Bird", "Tree", "Moon", "Sun",
    "Star", "Sea", "Hill", "Bridge", "Door", "Window", "Letter", "Book",
    "Dream", "Shadow", "Light", "Fire", "Rain", "Wind", "Stone", "Flower",
]
ENDINGS = [
    "of Lagos", "of Accra", "of Cairo", "of Nairobi", "of Kano",
    "of the River", "of the Desert", "of the Night", "of the Morning",
    "We Forgot", "We Remember", "That Waited", "That Spoke",
    "and the Rain", "and the Sea", "Without a Name", "With No Map",
]


def build_catalogue(n=500):
    titles, seen = [], set()
    for title in FAMOUS:
        clean = " ".join(title.split())
        if clean not in seen:
            seen.add(clean)
            titles.append(clean)
    i = 0
    while len(titles) < n:
        invented = f"{STARTERS[i % len(STARTERS)]} {MIDDLES[(i * 3) % len(MIDDLES)]} {ENDINGS[(i * 7) % len(ENDINGS)]}"
        if invented not in seen:
            seen.add(invented)
            titles.append(invented)
        i += 1
        if i > 20000:
            break
    return titles[:n]


BOOKS = build_catalogue(500)
print("Catalogue size:", len(BOOKS))
print("Unique titles: ", len(set(BOOKS)))
print("First 3:", BOOKS[:3])
print("Last 3: ", BOOKS[-3:])


Catalogue size: 500
Unique titles:  500
First 3: ['Pride and Prejudice', 'To Kill a Mockingbird', 'The Great Gatsby']
Last 3:  ['Old Tree of the Morning', 'New Star Without a Name', 'Lost Bridge of the River']


### Count with a plain dict + a set


In [25]:
def clean_word(raw):
    """Turn King's, into kings. Keep letters and digits only."""
    return "".join(ch for ch in raw.lower() if ch.isalnum())


def count_with_plain_dict(titles):
    counts = {}
    unique_words = set()
    for title in titles:
        for raw in title.split():
            word = clean_word(raw)
            if not word:
                continue
            unique_words.add(word)                 # set ignores duplicates
            if word in counts:                     # FAST membership (hash)
                counts[word] += 1
            else:
                counts[word] = 1
            # shorter: counts[word] = counts.get(word, 0) + 1
    return counts, unique_words


counts, unique = count_with_plain_dict(BOOKS)
print("Different words in the dict:", len(counts))
print("Different words in the set: ", len(unique))
print("These two numbers match:    ", len(counts) == len(unique))


Different words in the dict: 212
Different words in the set:  212
These two numbers match:     True


### Top words, rare words, and Counter as a check


In [26]:
from collections import Counter

ranked = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
print("TOP 15 WORDS IN THE TITLES")
for i, (word, n) in enumerate(ranked[:15], start=1):
    print(f"{i:2}. {word:12} {n:4}  {'#' * min(n, 40)}")

rare = {w for w, n in counts.items() if n == 1}
print()
print("Words that appear in only one title:", len(rare))
print("A few of them:", ", ".join(sorted(rare)[:12]))

all_words = []
for title in BOOKS:
    for raw in title.split():
        word = clean_word(raw)
        if word:
            all_words.append(word)
print()
print("Counter agrees on top word:", Counter(all_words).most_common(1))
print()
print("Instant dict questions:")
print("  counts.get('lagos', 0) =", counts.get("lagos", 0))
print("  'lagos' in counts      =", "lagos" in counts)
print("  winning word           =", max(counts, key=counts.get))


TOP 15 WORDS IN THE TITLES
 1. of            245  ########################################
 2. the           199  ########################################
 3. and            55  ########################################
 4. river          54  ########################################
 5. we             51  ########################################
 6. that           51  ########################################
 7. a              47  ########################################
 8. night          40  ########################################
 9. sea            39  #######################################
10. morning        39  #######################################
11. desert         38  ######################################
12. rain           38  ######################################
13. with           27  ###########################
14. no             26  ##########################
15. lagos          26  ##########################

Words that appear in only one title: 118
A few of them: aka

## 13. Question patterns (solve any related question)

Almost every dict/set question is one of these costumes. Name the costume, pick the structure, write 5–10 lines.

| Pattern | Idea | Structure |
| --- | --- | --- |
| **A Frequency** | count letters, words, votes | `d[x] = d.get(x, 0) + 1` |
| **B Seen** | “have I met this before?” | `set` |
| **C Complement / two-sum** | does the partner exist? | dict of value → index |
| **D Anagram** | same letters / same counts | count-dict or sorted signature |
| **E Grouping** | category → members | dict of lists |
| **F Set algebra** | overlap, banned words | intersection / union / difference |
| **G Cache / memo** | input → already-computed answer | dict |
| **H Graph** | node → neighbours | dict of sets/lists |

When you see a new question, ask:

1. Do I need a number attached to each thing? → **dict**
2. Do I only need yes/no membership or uniqueness? → **set**
3. Would a list scan inside a loop become O(n²)? → replace with set/dict
4. Is the key a list? → convert to **tuple** / **frozenset**


### Pattern A — frequency and first unique character


In [27]:
def word_count(text):
    counts = {}
    for word in text.lower().split():
        counts[word] = counts.get(word, 0) + 1
    return counts

print(word_count("the cat and the hat"))


def first_unique_char(s):
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    for ch in s:
        if counts[ch] == 1:
            return ch
    return None

print(first_unique_char("aabbcdde"))
print(first_unique_char("aabb"))


{'the': 2, 'cat': 1, 'and': 1, 'hat': 1}
c
None


### Pattern B — contains duplicate


In [28]:
def has_duplicate(nums):
    seen = set()
    for n in nums:
        if n in seen:
            return True
        seen.add(n)
    return False
    # one-liner: return len(nums) != len(set(nums))

print(has_duplicate([1, 2, 3, 1]))
print(has_duplicate([1, 2, 3]))


True
False


### Pattern C — two sum


In [29]:
def two_sum(nums, target):
    need = {}   # value we still need -> index of the partner
    for i, n in enumerate(nums):
        if n in need:
            return [need[n], i]
        need[target - n] = i
    return None

print(two_sum([2, 7, 11, 15], 9))


[0, 1]


### Pattern D — anagrams and group anagrams


In [30]:
def are_anagrams(a, b):
    if len(a) != len(b):
        return False
    counts = {}
    for ch in a:
        counts[ch] = counts.get(ch, 0) + 1
    for ch in b:
        if ch not in counts:
            return False
        counts[ch] -= 1
        if counts[ch] == 0:
            del counts[ch]
    return len(counts) == 0

print(are_anagrams("listen", "silent"))
print(are_anagrams("python", "typhonx"))


def group_anagrams(words):
    groups = {}
    for w in words:
        key = "".join(sorted(w))   # 'cba' and 'abc' share key 'abc'
        groups.setdefault(key, []).append(w)
    return list(groups.values())

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))


True
False
[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### Pattern E + F — grouping and set algebra (banned words)


In [31]:
def common_items(a, b):
    return list(set(a) & set(b))

print(common_items([1, 2, 2, 3], [2, 3, 4]))


def most_common_word(paragraph, banned):
    banned_set = set(word.lower() for word in banned)  # O(1) "is it banned?"
    counts = {}
    current = []
    for ch in paragraph.lower() + " ":
        if ch.isalpha():
            current.append(ch)
        else:
            if current:
                word = "".join(current)
                current = []
                if word not in banned_set:
                    counts[word] = counts.get(word, 0) + 1
    return max(counts, key=counts.get)

print(most_common_word(
    "Bob hit a ball, the hit BALL flew far after it was hit.",
    ["hit"],
))


[2, 3]
ball


### Pattern G — cache (memo)


In [32]:
fib_cache = {0: 0, 1: 1}

def fib(n):
    if n not in fib_cache:
        fib_cache[n] = fib(n - 1) + fib(n - 2)
    return fib_cache[n]

print(fib(20))
print("cache size:", len(fib_cache))


6765
cache size: 21


### Pattern H — graph as a dict


In [33]:
graph = {
    "Lagos": {"Ibadan", "Benin"},
    "Ibadan": {"Lagos", "Ilorin"},
    "Benin": {"Lagos", "Onitsha"},
    "Ilorin": {"Ibadan"},
    "Onitsha": {"Benin"},
}
print("neighbours of Lagos:", graph["Lagos"])
print("edge Lagos-Ibadan?", "Ibadan" in graph["Lagos"])


neighbours of Lagos: {'Benin', 'Ibadan'}
edge Lagos-Ibadan? True


## 14. Drill — say the answer before you run

Cover the output. Predict. Then `Shift+Enter`.


In [34]:
print("1.", type({}))
print("2.", type(set()))
print("3.", {1, 2, 3} & {3, 4})
print("4.", {"a": 1, "b": 2} == {"b": 2, "a": 1})
print("5.", "author" in {"title": "x", "author": "y"})
print("6.", {"the": 2, "cat": 1}.get("hat", 0))

# 7. What happens?
try:
    {"ok": 1}[[1, 2]] = 2
    print("7. no error")
except TypeError:
    print("7. TypeError — list is unhashable")

# 8. invert
print("8.", {v: k for k, v in {"Ada": 1, "Bola": 2}.items()})


1. <class 'dict'>
2. <class 'set'>
3. {3}
4. True
5. True
6. 0
7. TypeError — list is unhashable
8. {1: 'Ada', 2: 'Bola'}


### Extra drills (run after you have an answer in your head)

1. Why is `x in my_dict` fast, but `x in my_list` slow for huge data?
2. Why can a list not be a dictionary key?
3. Worst-case lookup for a dict?
4. Store a mapping from `(row, column)` to a chess piece — what key type?
5. In the 500-book project, why not 500 nested lists of words as the *main* structure?

**Answers**

1. Dict hashes `x` and jumps (average O(1)). List compares `x` to item 0, 1, 2… (O(n)).
2. Lists are mutable and unhashable. Use a tuple.
3. O(n) if everything collides. Average O(1) is what you quote in practice.
4. Tuple keys: `board[(0, 0)] = "rook"`.
5. You would rescan to answer “how many times is `the` used?”. A dict answers in one hash. Lists of titles are fine as **input**; the **index** of frequencies is a dict.


## 15. Mastery checklist

You are solid at this topic when you can do all of these without notes:

- [ ] Build, read, update, delete dicts. Use `get` safely.
- [ ] Build sets. Use `in`, `add`, `| & -`.
- [ ] Explain hashing with the locker story and say “average O(1)”.
- [ ] Say why lists are O(n) for membership.
- [ ] Pick dict vs set vs list for a new problem in 10 seconds.
- [ ] Know which types are hashable.
- [ ] Write the frequency loop in your sleep.
- [ ] Sort a dict by value.
- [ ] Group items into a dict of lists.
- [ ] Avoid the O(n²) `if x in list` trap.
- [ ] Explain the 500-book counter: input list of strings → dict of frequencies + set of unique words.
- [ ] Recognise the 8 patterns in section 13 and code them.
